# Topic Modeling of Patent-Cited and Policy-Cited Scholarly Literature

This notebook performs comparative topic modeling on two collections of
scholarly publications originating from the same broader Scopus research
corpus but identified through different forms of external citation:

1. **Patent-cited scholarly literature** — scholarly publications identified
   through Lens as being cited by patents.
2. **Policy-cited scholarly literature** — scholarly publications identified
   through Overton as being cited by policy documents.

The unit of analysis in both datasets is a **scholarly publication**. Patents
and policy documents themselves are not topic-modeled in this notebook.

The objective is to identify and compare the thematic structure of scholarly
research associated with patent uptake and policy uptake.

The analysis will include text preprocessing, document-term matrix
construction, LDA model selection, final topic estimation, topic
interpretation, prevalence analysis, cross-corpus thematic comparison, and
temporal analysis.

In [46]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import subprocess
import tempfile


# ---------------------------------------------------------
# Project directories
# ---------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR


DATA_DIR = (
    PROJECT_DIR
    / "data"
)

LDA_DATA_DIR = (
    DATA_DIR
    / "lda"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Project directory :",
    PROJECT_DIR
)

print(
    "Data directory    :",
    DATA_DIR
)

print(
    "LDA data directory:",
    LDA_DATA_DIR
)

print(
    "Output directory  :",
    OUTPUT_DIR
)

Project directory : m:\projects_latex\review_paper_2
Data directory    : m:\projects_latex\review_paper_2\data
LDA data directory: m:\projects_latex\review_paper_2\data\lda
Output directory  : m:\projects_latex\review_paper_2\output


## 1. Load Patent-Cited and Policy-Cited Scholarly Data

Two scholarly publication datasets are loaded for independent topic modeling.

The **patent-cited corpus** contains publications from the broader Scopus
research corpus that were identified through Lens as being cited by patents.

The **policy-cited corpus** contains publications from the broader Scopus
research corpus that were identified through Overton as being cited by policy
documents.

Because both datasets contain scholarly publications, the comparison focuses
on differences in the thematic composition of research associated with two
forms of external uptake: technological uptake through patents and policy
uptake through policy-document citations.

The two collections are retained as separate corpora throughout the
topic-modeling workflow.

In [47]:
# ---------------------------------------------------------
# Input files
# ---------------------------------------------------------

PATENT_CITED_FILE = (
    LDA_DATA_DIR
    / "lens_from_scopus_scholar.xlsx"
)

POLICY_CITED_FILE = (
    LDA_DATA_DIR
    / "overton_from_scopus_scholar.xlsx"
)


# ---------------------------------------------------------
# Validate files
# ---------------------------------------------------------

for input_file in [
    PATENT_CITED_FILE,
    POLICY_CITED_FILE,
]:

    if not input_file.exists():

        raise FileNotFoundError(
            f"Missing input file: "
            f"{input_file}"
        )


print(
    "Patent-cited file:",
    PATENT_CITED_FILE
)

print(
    "Policy-cited file:",
    POLICY_CITED_FILE
)

Patent-cited file: m:\projects_latex\review_paper_2\data\lda\lens_from_scopus_scholar.xlsx
Policy-cited file: m:\projects_latex\review_paper_2\data\lda\overton_from_scopus_scholar.xlsx


In [48]:
# ---------------------------------------------------------
# Load datasets
# ---------------------------------------------------------

patent_cited = pd.read_excel(
    PATENT_CITED_FILE
)

policy_cited = pd.read_excel(
    POLICY_CITED_FILE
)


# ---------------------------------------------------------
# Basic dimensions
# ---------------------------------------------------------

print(
    "PATENT-CITED SCHOLARLY LITERATURE"
)

print(
    "--------------------------------"
)

print(
    f"Documents : "
    f"{len(patent_cited):,}"
)

print(
    f"Columns   : "
    f"{len(patent_cited.columns):,}"
)


print()


print(
    "POLICY-CITED SCHOLARLY LITERATURE"
)

print(
    "--------------------------------"
)

print(
    f"Documents : "
    f"{len(policy_cited):,}"
)

print(
    f"Columns   : "
    f"{len(policy_cited.columns):,}"
)

PATENT-CITED SCHOLARLY LITERATURE
--------------------------------
Documents : 778
Columns   : 17

POLICY-CITED SCHOLARLY LITERATURE
--------------------------------
Documents : 392
Columns   : 20


### 1.1 Validate Topic-Modeling Fields

Before constructing the topic-modeling corpora, the input datasets are checked
for the fields required by the subsequent analysis.

`Title` and `Abstract` are required for document interpretation and text
modeling. `DOI` is used as the principal publication identifier where
available, while `Year` is required for temporal analysis.

The validation step also reports missing abstracts and duplicate DOI values
without modifying the datasets. Any deduplication or exclusion decision is
performed explicitly in a subsequent step.

In [49]:
# ---------------------------------------------------------
# Required fields
# ---------------------------------------------------------

REQUIRED_COLUMNS = [
    "Title",
    "Abstract",
    "DOI",
    "Year",
]


for corpus_name, corpus_df in {
    "patent_cited": patent_cited,
    "policy_cited": policy_cited,
}.items():

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in corpus_df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"{corpus_name} is missing "
            f"required columns: "
            f"{missing_columns}"
        )


print(
    "Required fields are available "
    "in both corpora."
)

Required fields are available in both corpora.


In [50]:
for corpus_name, corpus_df in {
    "patent_cited": patent_cited,
    "policy_cited": policy_cited,
}.items():

    display_name = (
        corpus_name
        .replace(
            "_",
            " "
        )
        .upper()
    )

    print()
    print(display_name)
    print("-" * len(display_name))

    print(
        f"Documents             : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles    : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts : "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts     : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

    print(
        f"Non-missing DOI       : "
        f"{corpus_df['DOI'].notna().sum():,}"
    )

    print(
        f"Unique DOI            : "
        f"{corpus_df['DOI'].dropna().nunique():,}"
    )

    year_numeric = pd.to_numeric(
        corpus_df["Year"],
        errors="coerce",
    )

    print(
        f"Non-missing years     : "
        f"{year_numeric.notna().sum():,}"
    )

    if year_numeric.notna().any():

        print(
            f"Year range            : "
            f"{int(year_numeric.min())}"
            f"-"
            f"{int(year_numeric.max())}"
        )


PATENT CITED
------------
Documents             : 778
Non-missing titles    : 778
Non-missing abstracts : 778
Missing abstracts     : 0
Non-missing DOI       : 778
Unique DOI            : 778
Non-missing years     : 778
Year range            : 1998-2026

POLICY CITED
------------
Documents             : 392
Non-missing titles    : 392
Non-missing abstracts : 392
Missing abstracts     : 0
Non-missing DOI       : 392
Unique DOI            : 392
Non-missing years     : 392
Year range            : 1998-2026


### 1.2 Define the Independent Topic-Modeling Corpora

The patent-cited and policy-cited scholarly collections are treated as
independent topic-modeling corpora.

Each corpus will receive its own preprocessing, document-term matrix,
vocabulary filtering, LDA model-selection procedure, final topic model, and
topic interpretation.

The number of latent topics is not assumed to be identical across the two
corpora. Topic identifiers are also not directly comparable between
independently estimated LDA models. For example, Topic 1 in the patent-cited
model does not necessarily correspond to Topic 1 in the policy-cited model.

If direct thematic comparison is required after topic interpretation, a
higher-level common thematic mapping can subsequently be constructed.

In [51]:
corpora = {
    "patent_cited":
        patent_cited.copy(),

    "policy_cited":
        policy_cited.copy(),
}


for corpus_name, corpus_df in (
    corpora.items()
):

    display_name = (
        corpus_name
        .replace(
            "_",
            " "
        )
        .upper()
    )

    print()
    print(display_name)
    print("-" * len(display_name))

    print(
        f"Documents          : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts: "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts  : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )


PATENT CITED
------------
Documents          : 778
Non-missing titles : 778
Non-missing abstracts: 778
Missing abstracts  : 0

POLICY CITED
------------
Documents          : 392
Non-missing titles : 392
Non-missing abstracts: 392
Missing abstracts  : 0


### 1.3 Analysis Output Directories

Generated files are stored separately from the source data.

The patent-cited and policy-cited analyses share a dedicated output hierarchy
for preprocessing results, LDA models, topic interpretation, comparative
analysis, temporal analysis, and figures.

Intermediate outputs are saved so that expensive preprocessing and LDA
estimation steps do not need to be repeated unnecessarily.

In [52]:
# ---------------------------------------------------------
# Patent-policy topic-modeling outputs
# ---------------------------------------------------------

ANALYSIS_OUTPUT_DIR = (
    OUTPUT_DIR
    / "patent_policy_topic_modeling"
)

R_PREPROCESS_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "r_preprocessing"
)

LDA_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "lda"
)

TOPIC_INTERPRETATION_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "topic_interpretation"
)

TOPIC_PREVALENCE_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "topic_prevalence"
)

COMPARISON_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "cross_corpus_comparison"
)

TEMPORAL_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "temporal_analysis"
)

FIGURE_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "figures"
)


for directory in [
    ANALYSIS_OUTPUT_DIR,
    R_PREPROCESS_DIR,
    LDA_DIR,
    TOPIC_INTERPRETATION_DIR,
    TOPIC_PREVALENCE_DIR,
    COMPARISON_DIR,
    TEMPORAL_DIR,
    FIGURE_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "Analysis output :",
    ANALYSIS_OUTPUT_DIR
)

print(
    "R preprocessing :",
    R_PREPROCESS_DIR
)

print(
    "LDA             :",
    LDA_DIR
)

print(
    "Interpretation  :",
    TOPIC_INTERPRETATION_DIR
)

print(
    "Prevalence      :",
    TOPIC_PREVALENCE_DIR
)

print(
    "Comparison      :",
    COMPARISON_DIR
)

print(
    "Temporal        :",
    TEMPORAL_DIR
)

print(
    "Figures         :",
    FIGURE_DIR
)

Analysis output : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling
R preprocessing : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\r_preprocessing
LDA             : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\lda
Interpretation  : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\topic_interpretation
Prevalence      : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\topic_prevalence
Comparison      : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\cross_corpus_comparison
Temporal        : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\temporal_analysis
Figures         : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\figures


## 2. Prepare Corpora for Topic Modeling

The patent-cited and policy-cited datasets contain complete titles,
abstracts, DOIs, and publication years, with one unique DOI per scholarly
publication.

Because both collections were derived from the same broader Scopus research
corpus and subsequently selected according to external citation status, no
additional domain-relevance filter is applied at this stage. Applying a new
content filter after citation-based selection could alter the composition of
the patent-cited and policy-cited samples.

For topic modeling, the abstract is used as the primary textual unit. A
stable document identifier is assigned to each publication so that the
processed text, LDA document-topic probabilities, representative documents,
and original publication metadata can be linked consistently throughout the
analysis.

In [53]:
# ---------------------------------------------------------
# Construct analysis corpora
# ---------------------------------------------------------

lda_corpora = {}

for corpus_name, corpus_df in (
    corpora.items()
):

    df = (
        corpus_df
        .copy()
        .reset_index(drop=True)
    )

    # Stable document identifier.
    # R uses 1-based indexing, so start at 1.
    df.insert(
        0,
        "document_id",
        np.arange(
            1,
            len(df) + 1,
        ),
    )

    # Ensure abstract is string and trimmed.
    df["Abstract"] = (
        df["Abstract"]
        .astype(str)
        .str.strip()
    )

    # Final safety check.
    valid_abstract = (
        df["Abstract"]
        .notna()
        & df["Abstract"].ne("")
    )

    if not valid_abstract.all():

        raise ValueError(
            f"{corpus_name} contains "
            f"{(~valid_abstract).sum()} "
            f"invalid abstracts."
        )

    lda_corpora[
        corpus_name
    ] = df


# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

for corpus_name, corpus_df in (
    lda_corpora.items()
):

    display_name = (
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print()
    print(display_name)
    print("-" * len(display_name))

    print(
        "Documents :",
        f"{len(corpus_df):,}"
    )

    print(
        "Document IDs:",
        f"{corpus_df['document_id'].min()}"
        f"-"
        f"{corpus_df['document_id'].max()}"
    )

    print(
        "Unique DOI:",
        f"{corpus_df['DOI'].nunique():,}"
    )


PATENT CITED
------------
Documents : 778
Document IDs: 1-778
Unique DOI: 778

POLICY CITED
------------
Documents : 392
Document IDs: 1-392
Unique DOI: 392


### 2.1 Save Topic-Modeling Metadata

A metadata table is saved for each corpus before text preprocessing.

The metadata files preserve the relationship between the sequential
`document_id` used by the topic-modeling workflow and the original
publication information. These files are subsequently used to recover
titles, DOIs, publication years, and other metadata when identifying
representative publications for each latent topic.

In [54]:
# ---------------------------------------------------------
# Metadata output directory
# ---------------------------------------------------------

METADATA_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# Save metadata
# ---------------------------------------------------------

for corpus_name, corpus_df in (
    lda_corpora.items()
):

    metadata_file = (
        METADATA_DIR
        / f"{corpus_name}_metadata.csv"
    )

    corpus_df.to_csv(
        metadata_file,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{corpus_name.upper():15s}: "
        f"{metadata_file}"
    )

PATENT_CITED   : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\metadata\patent_cited_metadata.csv
POLICY_CITED   : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\metadata\policy_cited_metadata.csv


## 3. R Text-Processing Backend

Text preprocessing is performed in R to ensure that the document-term
matrices used for LDA are constructed using the same text-processing
implementation as the topic-modeling backend.

The same preprocessing procedure is applied independently to the
patent-cited and policy-cited corpora.

The preprocessing pipeline includes:

1. conversion to lowercase;
2. punctuation removal;
3. number removal;
4. whitespace normalization;
5. English stopword removal;
6. removal of query-specific terms; and
7. English stemming.

Applying an identical preprocessing procedure to both corpora ensures that
differences observed in the resulting topic models are not caused by
different text-processing rules.

In [55]:
from pathlib import Path
import subprocess

RSCRIPT = Path(
    r"C:\Program Files\R\R-4.5.2\bin\Rscript.exe"
)

if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"Rscript not found: {RSCRIPT}"
    )

# Check required R packages.
r_package_check = subprocess.run(
    [
        str(RSCRIPT),
        "-e",
        (
            'pkgs <- c("tm", "SnowballC", "slam", "topicmodels"); '
            'ok <- sapply(pkgs, requireNamespace, quietly=TRUE); '
            'cat(paste(pkgs, ok, sep="="), sep="\\n")'
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print("Rscript:")
print(RSCRIPT)

print("\nRequired R packages:")
print(r_package_check.stdout)

Rscript:
C:\Program Files\R\R-4.5.2\bin\Rscript.exe

Required R packages:
tm=TRUE
SnowballC=TRUE
slam=TRUE
topicmodels=TRUE



### 3.1 Text-Preprocessing Configuration

The same text-preprocessing procedure is applied to the patent-cited and
policy-cited scholarly corpora to maintain methodological consistency.

The preprocessing pipeline converts text to lowercase, removes punctuation
and numbers, removes standard English stopwords and query-specific terms,
applies English stemming, and normalizes whitespace.

Query-specific terms are removed because the two citation-selected corpora
originate from the same broader Scopus search. Terms imposed by the original
retrieval query are therefore expected to occur broadly by construction and
provide limited thematic discrimination.

In [56]:
# ---------------------------------------------------------
# Shared preprocessing configuration
# ---------------------------------------------------------

MIN_TERM_LENGTH = 3

QUERY_STOPWORDS = [
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",
]

print(
    "Minimum term length :",
    MIN_TERM_LENGTH
)

print(
    "Query stopwords     :",
    QUERY_STOPWORDS
)

Minimum term length : 3
Query stopwords     : ['power', 'flow', 'machine', 'learning', 'optimization', 'optimisation']


### 3.2 Create Shared R Preprocessing Script

A single R preprocessing script is used for both corpora.

Using the same preprocessing implementation ensures that differences between
the patent-cited and policy-cited topic models are not introduced by
corpus-specific text-cleaning procedures.

The processed text retains the original `document_id`, allowing all later
DTM and LDA outputs to be linked back to the original scholarly publication.

In [57]:
R_PREPROCESS_SCRIPT = (
    R_PREPROCESS_DIR
    / "preprocess_patent_policy_corpora.R"
)

r_preprocess_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

input_file  <- args[1]
output_file <- args[2]

suppressPackageStartupMessages({
    library(tm)
    library(SnowballC)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

query_stopwords <- c(
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation"
)

# ------------------------------------------------------------
# Load corpus
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

if (!"document_id" %in% names(data)) {
    stop(
        "Input file does not contain document_id."
    )
}

if (!"Abstract" %in% names(data)) {
    stop(
        "Input file does not contain Abstract."
    )
}

abstracts <- data$Abstract

# ------------------------------------------------------------
# Construct corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        abstracts
    )
)

# ------------------------------------------------------------
# Text preprocessing
# ------------------------------------------------------------

corpus <- tm_map(
    corpus,
    content_transformer(tolower)
)

corpus <- tm_map(
    corpus,
    removePunctuation
)

corpus <- tm_map(
    corpus,
    removeNumbers
)

# Explicitly remove ©
corpus <- tm_map(
    corpus,
    content_transformer(
        function(x) {
            gsub(
                "\u00A9",
                " ",
                x,
                fixed = TRUE
            )
        }
    )
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    removeWords,
    stopwords("english")
)

corpus <- tm_map(
    corpus,
    removeWords,
    query_stopwords
)

corpus <- tm_map(
    corpus,
    stemDocument,
    language = "english"
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

# ------------------------------------------------------------
# Extract processed text
# ------------------------------------------------------------

processed_text <- vapply(
    corpus,
    as.character,
    character(1)
)

output <- data.frame(
    document_id = data$document_id,
    processed_text = processed_text,
    stringsAsFactors = FALSE
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

write.csv(
    output,
    output_file,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Documents processed:",
    nrow(output),
    "\n"
)

cat(
    "Empty processed documents:",
    sum(
        !nzchar(
            trimws(
                output$processed_text
            )
        )
    ),
    "\n"
)
'''

R_PREPROCESS_SCRIPT.write_text(
    r_preprocess_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_PREPROCESS_SCRIPT
)

Created: m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\r_preprocessing\preprocess_patent_policy_corpora.R


### 3.3 Preprocess Patent-Cited and Policy-Cited Abstracts

The abstract text from each corpus is passed independently through the shared
R preprocessing pipeline.

The resulting processed-text files are saved to disk so that later DTM and
LDA stages can be reproduced without repeating the text-preprocessing step.

In [58]:
processed_corpora = {}


for corpus_name, corpus_df in (
    lda_corpora.items()
):

    # -----------------------------------------------------
    # Files
    # -----------------------------------------------------

    input_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_abstracts.csv"
    )

    output_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    # -----------------------------------------------------
    # Save abstract input
    # -----------------------------------------------------

    corpus_df[
        [
            "document_id",
            "Abstract",
        ]
    ].to_csv(
        input_file,
        index=False,
        encoding="utf-8",
    )

    # -----------------------------------------------------
    # Run R
    # -----------------------------------------------------

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    preprocess_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_PREPROCESS_SCRIPT),
            str(input_file),
            str(output_file),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        preprocess_run.returncode
    )

    if preprocess_run.stdout.strip():

        print()
        print("R output:")

        print(
            preprocess_run.stdout
        )

    if preprocess_run.stderr.strip():

        print()
        print("R messages:")

        print(
            preprocess_run.stderr
        )

    if preprocess_run.returncode != 0:

        raise RuntimeError(
            f"R preprocessing failed "
            f"for {corpus_name}."
        )

    # -----------------------------------------------------
    # Reload processed corpus
    # -----------------------------------------------------

    processed_df = pd.read_csv(
        output_file
    )

    processed_corpora[
        corpus_name
    ] = processed_df


PATENT CITED
----------------------------------------
Return code: 0

R output:
Documents processed: 778 
Empty processed documents: 0 


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 


POLICY CITED
----------------------------------------
Return code: 0

R output:
Documents processed: 392 
Empty processed documents: 0 


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 



### 3.4 Validate Processed Corpora

The processed corpora are checked for document-count preservation and empty
documents.

The number and order of documents should remain unchanged during
preprocessing. Documents that become empty after stopword removal and
stemming would require special treatment before DTM construction.

In [59]:
for corpus_name, processed_df in (
    processed_corpora.items()
):

    expected_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    empty_mask = (
        processed_df[
            "processed_text"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    print(
        "Expected documents :",
        f"{expected_documents:,}"
    )

    print(
        "Processed documents:",
        f"{len(processed_df):,}"
    )

    print(
        "Empty documents    :",
        f"{empty_mask.sum():,}"
    )

    print(
        "Document IDs match :",
        processed_df[
            "document_id"
        ].tolist()
        ==
        lda_corpora[
            corpus_name
        ][
            "document_id"
        ].tolist()
    )


PATENT CITED
----------------------------------------
Expected documents : 778
Processed documents: 778
Empty documents    : 0
Document IDs match : True

POLICY CITED
----------------------------------------
Expected documents : 392
Processed documents: 392
Empty documents    : 0
Document IDs match : True


### 3.5 Inspect Frequent Processed Terms

The most frequent processed terms are inspected before final DTM
construction.

This diagnostic helps identify residual retrieval terms, database
boilerplate, generic scholarly language, or other high-frequency terms that
may dominate the latent topics without providing useful thematic
information.

The purpose of this step is diagnostic rather than to automatically remove
frequent terms. Additional stopwords should only be introduced when their
removal is substantively justified and applied consistently to both corpora.

In [60]:
from collections import Counter


TOP_N_TERMS = 30

frequent_term_tables = {}


for corpus_name, processed_df in (
    processed_corpora.items()
):

    tokens = []

    for text in (
        processed_df[
            "processed_text"
        ]
        .fillna("")
        .astype(str)
    ):

        tokens.extend(
            text.split()
        )

    counts = Counter(
        tokens
    )

    frequent_terms = pd.DataFrame(
        counts.most_common(
            TOP_N_TERMS
        ),
        columns=[
            "Term",
            "Frequency",
        ],
    )

    frequent_term_tables[
        corpus_name
    ] = frequent_terms

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 40
    )

    display(
        frequent_terms
    )


PATENT CITED


,Term,Frequency
0,system,1454
1,propos,1143
2,algorithm,1006
3,method,1006
4,model,983
5,energi,962
6,distribut,802
7,use,783
8,network,744
9,control,736



POLICY CITED


,Term,Frequency
0,energi,917
1,system,917
2,model,458
3,distribut,431
4,grid,424
5,use,395
6,oper,343
7,paper,342
8,method,340
9,electr,335


## 4. Document-Term Matrix Construction

Separate document-term matrices (DTMs) are constructed from the processed
patent-cited and policy-cited abstracts.

Each DTM represents scholarly publications as rows and processed vocabulary
terms as columns, with cell values corresponding to term frequencies.

Before constructing the final LDA input matrices, the raw vocabulary is
examined under alternative minimum document-frequency thresholds. This is
necessary because the patent-cited and policy-cited corpora differ in size
and are substantially smaller than the broader scholarly corpus from which
they were selected.

The document-frequency threshold is therefore selected from corpus-specific
diagnostics rather than inherited automatically from a previous analysis.

### 4.1 Raw Vocabulary Diagnostics

An initial DTM is constructed using all processed terms satisfying the
minimum term-length requirement.

For every vocabulary term, two quantities are recorded:

- **Term frequency (TF):** total number of occurrences across the corpus.
- **Document frequency (DF):** number of different documents containing
  the term.

The raw vocabulary diagnostics are subsequently used to evaluate alternative
rare-term filtering thresholds.

In [61]:
R_DTM_DIAGNOSTIC_SCRIPT = (
    R_PREPROCESS_DIR
    / "dtm_diagnostics_patent_policy.R"
)

r_dtm_diagnostic_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
summary_file   <- args[2]
terms_file     <- args[3]

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3

# ------------------------------------------------------------
# Load processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

# ------------------------------------------------------------
# Construct raw DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Frequencies
# ------------------------------------------------------------

document_frequency <- (
    slam::col_sums(
        dtm > 0
    )
)

term_frequency <- (
    slam::col_sums(
        dtm
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary <- data.frame(
    Documents = dtm$nrow,
    Vocabulary = dtm$ncol,
    Tokens = sum(dtm$v),
    Min_Term_Length = MIN_TERM_LENGTH
)

# ------------------------------------------------------------
# Term diagnostics
# ------------------------------------------------------------

terms <- data.frame(
    Term = dtm$dimnames$Terms,
    Document_Frequency = as.numeric(
        document_frequency
    ),
    Term_Frequency = as.numeric(
        term_frequency
    ),
    stringsAsFactors = FALSE
)

terms <- terms[
    order(
        -terms$Document_Frequency,
        -terms$Term_Frequency
    ),
]

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

write.csv(
    summary,
    summary_file,
    row.names = FALSE
)

write.csv(
    terms,
    terms_file,
    row.names = FALSE
)

cat(
    "Documents:",
    dtm$nrow,
    "\n"
)

cat(
    "Raw vocabulary:",
    dtm$ncol,
    "\n"
)

cat(
    "Tokens:",
    sum(dtm$v),
    "\n"
)
'''

R_DTM_DIAGNOSTIC_SCRIPT.write_text(
    r_dtm_diagnostic_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_DTM_DIAGNOSTIC_SCRIPT
)

Created: m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\r_preprocessing\dtm_diagnostics_patent_policy.R


In [62]:
raw_dtm_summaries = {}
raw_dtm_terms = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_raw_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_raw_dtm_terms.csv"
    )

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    dtm_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_DIAGNOSTIC_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        dtm_run.returncode
    )

    if dtm_run.stdout.strip():

        print()
        print("R output:")

        print(
            dtm_run.stdout
        )

    if dtm_run.stderr.strip():

        print()
        print("R messages:")

        print(
            dtm_run.stderr
        )

    if dtm_run.returncode != 0:

        raise RuntimeError(
            f"DTM diagnostics failed "
            f"for {corpus_name}."
        )

    raw_dtm_summaries[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )

    raw_dtm_terms[
        corpus_name
    ] = pd.read_csv(
        terms_file
    )


PATENT CITED
----------------------------------------
Return code: 0

R output:
Documents: 778 
Raw vocabulary: 6277 
Tokens: 93133 


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 


POLICY CITED
----------------------------------------
Return code: 0

R output:
Documents: 392 
Raw vocabulary: 4088 
Tokens: 47199 


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 



### 4.2 Minimum Document-Frequency Diagnostics

Terms occurring in very few documents can increase DTM sparsity and
introduce corpus-specific noise into LDA.

Alternative minimum document-frequency thresholds are therefore evaluated
before constructing the final DTMs.

Because the two corpora differ in size, thresholds are expressed as
percentages of the corresponding corpus size. This preserves a comparable
relative filtering rule while allowing the absolute minimum DF to differ
between the patent-cited and policy-cited corpora.

No threshold is selected automatically. The retained vocabulary under each
candidate threshold is inspected before fixing the final configuration.

In [63]:
import math


DF_PERCENTAGES = [
    0.005,   # 0.5%
    0.010,   # 1.0%
    0.015,   # 1.5%
    0.020,   # 2.0%
    0.025,   # 2.5%
    0.030,   # 3.0%
]


df_diagnostic_tables = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    terms = (
        raw_dtm_terms[
            corpus_name
        ]
        .copy()
    )

    n_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    rows = []

    for fraction in DF_PERCENTAGES:

        minimum_df = math.ceil(
            n_documents
            * fraction
        )

        retained = (
            terms[
                "Document_Frequency"
            ]
            >= minimum_df
        )

        rows.append({
            "DF_Percent":
                fraction * 100,

            "Minimum_DF":
                minimum_df,

            "Retained_Terms":
                int(
                    retained.sum()
                ),

            "Removed_Terms":
                int(
                    (~retained).sum()
                ),

            "Vocabulary_Retained_Percent":
                (
                    100
                    * retained.sum()
                    / len(terms)
                ),
        })

    diagnostics = pd.DataFrame(
        rows
    )

    df_diagnostic_tables[
        corpus_name
    ] = diagnostics

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 40
    )

    display(
        diagnostics
    )


PATENT CITED


,DF_Percent,Minimum_DF,Retained_Terms,Removed_Terms,Vocabulary_Retained_Percent
0,0.5,4,1899,4378,30.253306
1,1.0,8,1295,4982,20.630875
2,1.5,12,999,5278,15.915246
3,2.0,16,831,5446,13.238808
4,2.5,20,712,5565,11.342998
5,3.0,24,622,5655,9.909192



POLICY CITED


,DF_Percent,Minimum_DF,Retained_Terms,Removed_Terms,Vocabulary_Retained_Percent
0,0.5,2,2147,1941,52.519569
1,1.0,4,1371,2717,33.537182
2,1.5,6,1040,3048,25.440313
3,2.0,8,866,3222,21.183953
4,2.5,10,735,3353,17.979452
5,3.0,12,641,3447,15.680039


In [64]:
for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    display(
        raw_dtm_summaries[
            corpus_name
        ]
    )


PATENT CITED


,Documents,Vocabulary,Tokens,Min_Term_Length
0,778,6277,93133,3



POLICY CITED


,Documents,Vocabulary,Tokens,Min_Term_Length
0,392,4088,47199,3


### 4.3 Final Document-Frequency Threshold

Based on the vocabulary-retention diagnostics, a minimum document-frequency threshold of **1% of the corresponding corpus size** is selected for the final topic-modeling vocabulary.

Using a proportional threshold applies the same relative filtering criterion to both corpora while allowing the absolute minimum document frequency to reflect differences in corpus size.

For corpus \(c\), the minimum document frequency is defined as:

$$
DF_{\min,c}
=
\left\lceil 0.01 N_c \right\rceil
$$

where \(N_c\) is the number of documents in corpus \(c\).

For the two corpora analyzed here, this gives:

$$
DF_{\min,\mathrm{patent}}
=
\left\lceil 0.01 \times 778 \right\rceil
=
8
$$

and

$$
DF_{\min,\mathrm{policy}}
=
\left\lceil 0.01 \times 392 \right\rceil
=
4.
$$

Therefore, a term must occur in at least **8 patent-cited publications** or **4 policy-cited publications** to be retained in the corresponding final DTM.

This filtering removes terms that occur in only a very small number of documents while retaining sufficient vocabulary for subsequent topic discovery. The threshold is specific to the present corpora and should be reassessed when the workflow is applied to datasets of different sizes or compositions.

In [65]:
FINAL_MIN_DOC_PERCENT = 0.01

final_min_doc_freq = {}

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    n_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    min_df = math.ceil(
        n_documents
        * FINAL_MIN_DOC_PERCENT
    )

    final_min_doc_freq[
        corpus_name
    ] = min_df

    print(
        f"{corpus_name.upper():15s} | "
        f"Documents: {n_documents:,} | "
        f"Minimum DF: {min_df}"
    )

PATENT_CITED    | Documents: 778 | Minimum DF: 8
POLICY_CITED    | Documents: 392 | Minimum DF: 4


### 4.4 Construct Final Document-Term Matrices

The final document-term matrix for each corpus is reconstructed from the
processed abstracts using the selected corpus-specific minimum
document-frequency threshold.

Terms not meeting the threshold are removed before LDA estimation.

Documents that become empty after vocabulary filtering are identified
explicitly. The final DTM summary records the number of documents,
vocabulary size, token count, minimum document frequency, and number of
empty documents.

In [67]:
R_FINAL_DTM_SCRIPT = (
    R_PREPROCESS_DIR
    / "construct_final_dtm_patent_policy.R"
)

r_final_dtm_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
summary_file   <- args[2]
terms_file     <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

# ------------------------------------------------------------
# Construct raw DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply document-frequency filtering
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

keep_terms <- (
    document_frequency
    >= MIN_DOC_FREQ
)

dtm <- dtm[
    ,
    keep_terms
]

# ------------------------------------------------------------
# Check empty documents
# ------------------------------------------------------------

document_tokens <- slam::row_sums(
    dtm
)

empty_documents <- (
    document_tokens == 0
)

n_empty <- sum(
    empty_documents
)

# ------------------------------------------------------------
# Final frequencies
# ------------------------------------------------------------

final_document_frequency <- (
    slam::col_sums(
        dtm > 0
    )
)

final_term_frequency <- (
    slam::col_sums(
        dtm
    )
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary <- data.frame(
    Documents = dtm$nrow,
    Vocabulary = dtm$ncol,
    Tokens = sum(dtm$v),
    Minimum_DF = MIN_DOC_FREQ,
    Min_Term_Length = MIN_TERM_LENGTH,
    Empty_Documents = n_empty
)

terms <- data.frame(
    Term = dtm$dimnames$Terms,
    Document_Frequency = as.numeric(
        final_document_frequency
    ),
    Term_Frequency = as.numeric(
        final_term_frequency
    ),
    stringsAsFactors = FALSE
)

terms <- terms[
    order(
        -terms$Document_Frequency,
        -terms$Term_Frequency
    ),
]

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

write.csv(
    summary,
    summary_file,
    row.names = FALSE
)

write.csv(
    terms,
    terms_file,
    row.names = FALSE
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Documents:",
    dtm$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm$ncol,
    "\n"
)

cat(
    "Tokens:",
    sum(dtm$v),
    "\n"
)

cat(
    "Minimum DF:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty documents:",
    n_empty,
    "\n"
)
'''

R_FINAL_DTM_SCRIPT.write_text(
    r_final_dtm_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_FINAL_DTM_SCRIPT
)

Created: m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\r_preprocessing\construct_final_dtm_patent_policy.R


In [68]:
final_dtm_summaries = {}
final_dtm_terms = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_terms.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    print(
        "Minimum DF:",
        min_df
    )

    final_dtm_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_FINAL_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        final_dtm_run.returncode
    )

    if final_dtm_run.stdout.strip():

        print()
        print("R output:")

        print(
            final_dtm_run.stdout
        )

    if final_dtm_run.stderr.strip():

        print()
        print("R messages:")

        print(
            final_dtm_run.stderr
        )

    if final_dtm_run.returncode != 0:

        raise RuntimeError(
            f"Final DTM construction failed "
            f"for {corpus_name}."
        )

    final_dtm_summaries[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )

    final_dtm_terms[
        corpus_name
    ] = pd.read_csv(
        terms_file
    )


PATENT CITED
----------------------------------------
Minimum DF: 8
Return code: 0

R output:
Documents: 778 
Vocabulary: 1295 
Tokens: 80923 
Minimum DF: 8 
Empty documents: 0 


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 


POLICY CITED
----------------------------------------
Minimum DF: 4
Return code: 0

R output:
Documents: 392 
Vocabulary: 1371 
Tokens: 42397 
Minimum DF: 4 
Empty documents: 0 


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 



### 4.5 Final DTM Summary

The final DTM dimensions and token counts are inspected before proceeding to
LDA model selection.

The principal checks are:

- all intended documents remain represented;
- no document becomes empty after vocabulary filtering;
- the retained vocabulary is sufficiently large for thematic discovery; and
- the relative vocabulary-filtering rule has been applied consistently to
  both corpora.

In [69]:
for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "=" * 40
    )

    display(
        final_dtm_summaries[
            corpus_name
        ]
    )


PATENT CITED


,Documents,Vocabulary,Tokens,Minimum_DF,Min_Term_Length,Empty_Documents
0,778,1295,80923,8,3,0



POLICY CITED


,Documents,Vocabulary,Tokens,Minimum_DF,Min_Term_Length,Empty_Documents
0,392,1371,42397,4,3,0


## 5. LDA Topic Modeling and Model Selection

Separate Latent Dirichlet Allocation (LDA) models are estimated for the
patent-cited and policy-cited scholarly corpora.

Because the two corpora may contain different thematic structures, the
number of latent topics is selected independently for each corpus rather
than imposing a common topic number.

Model selection is performed in stages. First, each corpus is divided into
training and held-out subsets using a reproducible random split. A coarse
search over candidate topic numbers is then performed using relatively short
Gibbs-sampling chains and evaluated using held-out perplexity.

Candidate topic resolutions identified during the screening stage are
subsequently evaluated using longer chains and additional diagnostics,
including topic coherence, inter-topic similarity, and manual
interpretability.

The final number of topics is therefore selected using both quantitative
model diagnostics and substantive inspection of the resulting topics.

### 5.1 Reproducible Training and Held-Out Splits

Each corpus is divided into an 80% training subset and a 20% held-out subset.

The training documents are used to estimate candidate LDA models, while the
held-out documents are used to evaluate predictive performance through
perplexity.

A fixed random seed is used so that the same document split is retained
throughout all candidate-model evaluations.

In [70]:
# ---------------------------------------------------------
# LDA model-selection configuration
# ---------------------------------------------------------

TRAIN_FRACTION = 0.80
RANDOM_SEED = 123

K_COARSE = [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
]

print(
    "Training fraction :",
    TRAIN_FRACTION
)

print(
    "Random seed       :",
    RANDOM_SEED
)

print(
    "Coarse K values   :",
    K_COARSE
)

Training fraction : 0.8
Random seed       : 123
Coarse K values   : [5, 10, 15, 20, 25, 30, 40]


In [72]:
# ---------------------------------------------------------
# Create reproducible train / held-out splits
# ---------------------------------------------------------

train_test_indices = {}

for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # Independent reproducible RNG for each corpus.
    rng = np.random.default_rng(
        RANDOM_SEED
    )

    n_documents = len(
        lda_corpora[
            corpus_name
        ]
    )

    # -----------------------------------------------------
    # Random permutation
    # -----------------------------------------------------

    all_indices = np.arange(
        1,
        n_documents + 1,
    )

    shuffled = rng.permutation(
        all_indices
    )

    n_train = int(
        np.floor(
            TRAIN_FRACTION
            * n_documents
        )
    )

    train_id = np.sort(
        shuffled[
            :n_train
        ]
    )

    test_id = np.sort(
        shuffled[
            n_train:
        ]
    )

    train_test_indices[
        corpus_name
    ] = {
        "train": train_id,
        "test": test_id,
    }

    # -----------------------------------------------------
    # Save
    # -----------------------------------------------------

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    pd.DataFrame({
        "train_id": train_id
    }).to_csv(
        train_file,
        index=False,
    )

    pd.DataFrame({
        "test_id": test_id
    }).to_csv(
        test_file,
        index=False,
    )

    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------

    print()

    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 40
    )

    print(
        "Total documents   :",
        f"{n_documents:,}"
    )

    print(
        "Training documents:",
        f"{len(train_id):,}"
    )

    print(
        "Held-out documents:",
        f"{len(test_id):,}"
    )

    print(
        "Overlap           :",
        len(
            set(train_id)
            & set(test_id)
        )
    )

    print(
        "Complete coverage :",
        len(
            set(train_id)
            | set(test_id)
        )
        == n_documents
    )


PATENT CITED
----------------------------------------
Total documents   : 778
Training documents: 622
Held-out documents: 156
Overlap           : 0
Complete coverage : True

POLICY CITED
----------------------------------------
Total documents   : 392
Training documents: 313
Held-out documents: 79
Overlap           : 0
Complete coverage : True


### 5.2 Coarse Topic-Number Search

A coarse topic-number search is performed independently for the patent-cited
and policy-cited corpora.

For each candidate number of topics \(K\), an LDA model is estimated using
the training documents and evaluated on the held-out documents using
perplexity.

Lower held-out perplexity indicates better predictive performance. However,
perplexity is used only as an initial screening criterion and does not by
itself determine the final number of topics.

Relatively short Gibbs-sampling chains are used during this stage to keep
the screening computationally efficient. Candidate topic resolutions
identified here will subsequently be evaluated using longer chains,
additional topic-quality diagnostics, and manual interpretability.

In [73]:
# ---------------------------------------------------------
# Coarse LDA search configuration
# ---------------------------------------------------------

COARSE_BURN_IN = 50
COARSE_ITERATIONS = 100
COARSE_THIN = 10

print(
    "K values   :",
    K_COARSE
)

print(
    "Burn-in    :",
    COARSE_BURN_IN
)

print(
    "Iterations :",
    COARSE_ITERATIONS
)

print(
    "Thin       :",
    COARSE_THIN
)

K values   : [5, 10, 15, 20, 25, 30, 40]
Burn-in    : 50
Iterations : 100
Thin       : 10


#### 5.2.1 Coarse-Search R Backend

The coarse-search backend reconstructs the filtered DTM for each corpus,
applies the previously saved training/held-out split, estimates an LDA model
for each candidate value of \(K\), and calculates held-out perplexity.

The vocabulary is determined from the training data before model estimation,
and documents that become empty after training-vocabulary alignment are
reported explicitly.

Results are written to disk after every completed candidate model so that
completed runs are preserved if execution is interrupted.

In [74]:
R_COARSE_SEARCH_SCRIPT = (
    LDA_DIR
    / "coarse_k_search_patent_policy.R"
)

r_coarse_search_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
train_file     <- args[2]
test_file      <- args[3]
output_file    <- args[4]
min_doc_freq   <- as.integer(args[5])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_TERM_LENGTH <- 3

K_VALUES <- c(
    5,
    10,
    15,
    20,
    25,
    30,
    40
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10
RANDOM_SEED <- 123

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

if (any(is.na(processed))) {
    stop(
        "Processed corpus contains missing text."
    )
}

# ------------------------------------------------------------
# Reconstruct raw DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        processed
    )
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply final corpus-level DF filter
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

keep_terms <- (
    document_frequency
    >= min_doc_freq
)

dtm <- dtm[
    ,
    keep_terms
]

# ------------------------------------------------------------
# Load fixed train/test indices
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- read.csv(
    test_file
)$test_id

# Validate split.
if (
    length(
        intersect(
            train_id,
            test_id
        )
    ) > 0
) {
    stop(
        "Training and test indices overlap."
    )
}

if (
    length(
        union(
            train_id,
            test_id
        )
    ) != dtm$nrow
) {
    stop(
        "Training/test indices do not cover all documents."
    )
}

# ------------------------------------------------------------
# Split DTM
# ------------------------------------------------------------

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# ------------------------------------------------------------
# Restrict to terms occurring in training data
# ------------------------------------------------------------

training_term_mask <- (
    slam::col_sums(
        dtm_train
    ) > 0
)

dtm_train <- dtm_train[
    ,
    training_term_mask
]

dtm_test <- dtm_test[
    ,
    training_term_mask
]

# ------------------------------------------------------------
# Remove empty documents if necessary
# ------------------------------------------------------------

train_nonempty <- (
    slam::row_sums(
        dtm_train
    ) > 0
)

test_nonempty <- (
    slam::row_sums(
        dtm_test
    ) > 0
)

empty_train <- sum(
    !train_nonempty
)

empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n"
)

cat(
    "Empty training removed:",
    empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Candidate K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = RANDOM_SEED,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            model,
            newdata = dtm_test
        )
    )

    elapsed_seconds <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity =
                heldout_perplexity,
            Elapsed_seconds =
                elapsed_seconds
        )
    )

    # Save incrementally.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed_seconds
        )
    )

    flush.console()
}
'''

R_COARSE_SEARCH_SCRIPT.write_text(
    r_coarse_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_COARSE_SEARCH_SCRIPT
)

Created: m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\lda\coarse_k_search_patent_policy.R


In [75]:
coarse_results = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    output_file = (
        LDA_DIR
        / f"{corpus_name}_coarse_k_search.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    print()
    print(
        "Running coarse K search..."
    )

    print(
        "Corpus     :",
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "Documents  :",
        f"{len(lda_corpora[corpus_name]):,}"
    )

    print(
        "Minimum DF :",
        min_df
    )

    print(
        "K values   :",
        K_COARSE
    )

    coarse_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_COARSE_SEARCH_SCRIPT),
            str(processed_file),
            str(train_file),
            str(test_file),
            str(output_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        coarse_run.returncode
    )

    if coarse_run.stdout.strip():

        print()
        print("R output:")

        print(
            coarse_run.stdout
        )

    if coarse_run.stderr.strip():

        print()
        print("R messages:")

        print(
            coarse_run.stderr
        )

    if coarse_run.returncode != 0:

        raise RuntimeError(
            f"Coarse K search failed "
            f"for {corpus_name}."
        )

    coarse_results[
        corpus_name
    ] = pd.read_csv(
        output_file
    )

    print()
    display(
        coarse_results[
            corpus_name
        ]
    )


Running coarse K search...
Corpus     : PATENT CITED
Documents  : 778
Minimum DF : 8
K values   : [5, 10, 15, 20, 25, 30, 40]
Return code: 0

R output:
Training documents: 622 
Held-out documents: 156 
Vocabulary: 1295 
Minimum DF: 8 
Empty training removed: 0 
Empty held-out removed: 0 

Fitting K = 5 ... perplexity = 574.9083 | 0.61 s
Fitting K = 10 ... perplexity = 535.4664 | 0.79 s
Fitting K = 15 ... perplexity = 516.2660 | 0.93 s
Fitting K = 20 ... perplexity = 502.5953 | 1.10 s
Fitting K = 25 ... perplexity = 497.7938 | 1.31 s
Fitting K = 30 ... perplexity = 490.2154 | 1.48 s
Fitting K = 40 ... perplexity = 475.0116 | 1.86 s


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 
4: package 'topicmodels' was built under R version 4.5.3 




,K,Perplexity,Elapsed_seconds
0,5,574.908263,0.607629
1,10,535.466407,0.786315
2,15,516.266023,0.932295
3,20,502.595318,1.097218
4,25,497.793754,1.312999
5,30,490.215411,1.479508
6,40,475.011638,1.859220



Running coarse K search...
Corpus     : POLICY CITED
Documents  : 392
Minimum DF : 4
K values   : [5, 10, 15, 20, 25, 30, 40]
Return code: 0

R output:
Training documents: 313 
Held-out documents: 79 
Vocabulary: 1371 
Minimum DF: 4 
Empty training removed: 0 
Empty held-out removed: 0 

Fitting K = 5 ... perplexity = 635.1950 | 0.41 s
Fitting K = 10 ... perplexity = 599.5869 | 0.68 s
Fitting K = 15 ... perplexity = 587.4517 | 0.79 s
Fitting K = 20 ... perplexity = 563.1485 | 0.89 s
Fitting K = 25 ... perplexity = 556.7617 | 1.06 s
Fitting K = 30 ... perplexity = 545.7692 | 1.20 s
Fitting K = 40 ... perplexity = 532.6884 | 1.26 s


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 
4: package 'topicmodels' was built under R version 4.5.3 




,K,Perplexity,Elapsed_seconds
0,5,635.194999,0.408754
1,10,599.586871,0.679147
2,15,587.451705,0.787133
3,20,563.148505,0.894664
4,25,556.761731,1.059482
5,30,545.769204,1.198715
6,40,532.688379,1.260552


### 5.3 Extended Topic-Number Search

The coarse search shows a continued reduction in held-out perplexity through
the largest tested topic resolution, \(K=40\), for both corpora.

Because no clear predictive plateau is observed within the initial search
range, the screening is extended to moderately larger topic resolutions.

The extended search is intentionally limited because the patent-cited and
policy-cited corpora contain substantially fewer documents than the broader
scholarly corpus. Increasing \(K\) excessively may produce highly fragmented
or weakly supported topics even when held-out perplexity continues to
decrease.

The extended search therefore evaluates \(K=50\) and \(K=60\). These results
are used only to characterize the continued perplexity trend. Final
topic-number selection will additionally consider semantic coherence,
inter-topic similarity, and substantive interpretability.

In [76]:
K_EXTENDED = [
    50,
    60,
]

print(
    "Extended K values:",
    K_EXTENDED
)

Extended K values: [50, 60]


#### 5.3.1 Extended-Search R Backend

The extended search uses the same training and held-out document splits,
vocabulary-filtering rules, random seed, and Gibbs-sampling configuration
as the coarse search.

Only the additional candidate topic resolutions are estimated, avoiding
unnecessary refitting of the models already evaluated during the coarse
search.

Results are saved separately for each corpus and later combined with the
coarse-search results.

In [77]:
R_EXTENDED_SEARCH_SCRIPT = (
    LDA_DIR
    / "extended_k_search_patent_policy.R"
)

r_extended_search_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
train_file     <- args[2]
test_file      <- args[3]
output_file    <- args[4]
min_doc_freq   <- as.integer(args[5])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_TERM_LENGTH <- 3

K_VALUES <- c(
    50,
    60
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10
RANDOM_SEED <- 123

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

if (any(is.na(processed))) {
    stop(
        "Processed corpus contains missing text."
    )
}

# ------------------------------------------------------------
# Reconstruct DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        processed
    )
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply final corpus-level DF filter
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

keep_terms <- (
    document_frequency
    >= min_doc_freq
)

dtm <- dtm[
    ,
    keep_terms
]

# ------------------------------------------------------------
# Load fixed train/test indices
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- read.csv(
    test_file
)$test_id

# ------------------------------------------------------------
# Validate split
# ------------------------------------------------------------

if (
    length(
        intersect(
            train_id,
            test_id
        )
    ) > 0
) {
    stop(
        "Training and test indices overlap."
    )
}

if (
    length(
        union(
            train_id,
            test_id
        )
    ) != dtm$nrow
) {
    stop(
        "Training/test indices do not cover all documents."
    )
}

# ------------------------------------------------------------
# Split DTM
# ------------------------------------------------------------

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# ------------------------------------------------------------
# Retain terms represented in training data
# ------------------------------------------------------------

training_term_mask <- (
    slam::col_sums(
        dtm_train
    ) > 0
)

dtm_train <- dtm_train[
    ,
    training_term_mask
]

dtm_test <- dtm_test[
    ,
    training_term_mask
]

# ------------------------------------------------------------
# Remove empty documents if necessary
# ------------------------------------------------------------

train_nonempty <- (
    slam::row_sums(
        dtm_train
    ) > 0
)

test_nonempty <- (
    slam::row_sums(
        dtm_test
    ) > 0
)

empty_train <- sum(
    !train_nonempty
)

empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n"
)

cat(
    "Empty training removed:",
    empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Extended K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = RANDOM_SEED,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            model,
            newdata = dtm_test
        )
    )

    elapsed_seconds <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity =
                heldout_perplexity,
            Elapsed_seconds =
                elapsed_seconds
        )
    )

    # Save after each completed K.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed_seconds
        )
    )

    flush.console()
}
'''

R_EXTENDED_SEARCH_SCRIPT.write_text(
    r_extended_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_EXTENDED_SEARCH_SCRIPT
)

Created: m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\lda\extended_k_search_patent_policy.R


#### 5.3.2 Run Extended Topic-Number Search

The additional topic resolutions are estimated independently for the
patent-cited and policy-cited corpora.

Only \(K=50\) and \(K=60\) are evaluated in this stage because the models
for \(K=5\) through \(K=40\) have already been estimated during the coarse
search.

In [80]:
extended_results = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    output_file = (
        LDA_DIR
        / f"{corpus_name}_extended_k_search.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    print()
    print(
        "Running extended K search..."
    )

    print(
        "Corpus     :",
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "Documents  :",
        f"{len(lda_corpora[corpus_name]):,}"
    )

    print(
        "Minimum DF :",
        min_df
    )

    print(
        "K values   :",
        K_EXTENDED
    )

    extended_run = subprocess.run(
        [
            str(RSCRIPT),
            str(R_EXTENDED_SEARCH_SCRIPT),
            str(processed_file),
            str(train_file),
            str(test_file),
            str(output_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        extended_run.returncode
    )

    if extended_run.stdout.strip():

        print()
        print("R output:")

        print(
            extended_run.stdout
        )

    if extended_run.stderr.strip():

        print()
        print("R messages:")

        print(
            extended_run.stderr
        )

    if extended_run.returncode != 0:

        raise RuntimeError(
            f"Extended K search failed "
            f"for {corpus_name}."
        )

    extended_results[
        corpus_name
    ] = pd.read_csv(
        output_file
    )

    print()

    display(
        extended_results[
            corpus_name
        ]
    )


Running extended K search...
Corpus     : PATENT CITED
Documents  : 778
Minimum DF : 8
K values   : [50, 60]
Return code: 0

R output:
Training documents: 622 
Held-out documents: 156 
Vocabulary: 1295 
Minimum DF: 8 
Empty training removed: 0 
Empty held-out removed: 0 

Fitting K = 50 ... perplexity = 457.5349 | 2.23 s
Fitting K = 60 ... perplexity = 457.5294 | 5.46 s


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 
4: package 'topicmodels' was built under R version 4.5.3 




,K,Perplexity,Elapsed_seconds
0,50,457.534926,2.233350
1,60,457.529385,5.458385



Running extended K search...
Corpus     : POLICY CITED
Documents  : 392
Minimum DF : 4
K values   : [50, 60]
Return code: 0

R output:
Training documents: 313 
Held-out documents: 79 
Vocabulary: 1371 
Minimum DF: 4 
Empty training removed: 0 
Empty held-out removed: 0 

Fitting K = 50 ... perplexity = 527.9837 | 1.33 s
Fitting K = 60 ... perplexity = 523.5859 | 2.76 s


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 
4: package 'topicmodels' was built under R version 4.5.3 




,K,Perplexity,Elapsed_seconds
0,50,527.983674,1.329229
1,60,523.585932,2.764421


#### 5.4.1 Candidate-Evaluation Backend

Each candidate LDA model is refitted using a longer Gibbs-sampling chain than
was used during topic-number screening.

For every candidate \(K\), the evaluation records held-out perplexity,
topic-level semantic coherence, and pairwise topic similarity.

Semantic coherence is calculated from the document co-occurrence frequencies
of the highest-probability words in each topic. Inter-topic similarity is
calculated from cosine similarity between topic-word probability
distributions.

The complete topic-level diagnostics are saved in addition to the
model-level summaries so that poorly performing individual topics can be
identified during interpretation.

In [86]:
R_CANDIDATE_EVALUATION_SCRIPT = (
    LDA_DIR
    / "candidate_evaluation_patent_policy.R"
)

r_candidate_evaluation_code = r'''
args <- commandArgs(
    trailingOnly = TRUE
)

processed_file <- args[1]
train_file     <- args[2]
test_file      <- args[3]
summary_file   <- args[4]
topic_file     <- args[5]
top_terms_file <- args[6]
min_doc_freq   <- as.integer(args[7])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_TERM_LENGTH <- 3

K_VALUES <- c(
    20,
    30,
    40,
    50
)

BURN_IN <- 500
ITERATIONS <- 2000
THIN <- 50

RANDOM_SEED <- 123

TOP_N <- 10

# ------------------------------------------------------------
# Load processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

if (any(is.na(processed))) {
    stop(
        "Processed corpus contains missing text."
    )
}

# ------------------------------------------------------------
# Construct DTM
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(
        processed
    )
)

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Apply final document-frequency filter
# ------------------------------------------------------------

document_frequency <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    document_frequency >= min_doc_freq
]

# ------------------------------------------------------------
# Fixed train/test split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- read.csv(
    test_file
)$test_id

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# ------------------------------------------------------------
# Restrict vocabulary to training data
# ------------------------------------------------------------

training_term_mask <- (
    slam::col_sums(
        dtm_train
    ) > 0
)

dtm_train <- dtm_train[
    ,
    training_term_mask
]

dtm_test <- dtm_test[
    ,
    training_term_mask
]

# ------------------------------------------------------------
# Remove empty documents
# ------------------------------------------------------------

train_nonempty <- (
    slam::row_sums(
        dtm_train
    ) > 0
)

test_nonempty <- (
    slam::row_sums(
        dtm_test
    ) > 0
)

n_empty_train <- sum(
    !train_nonempty
)

n_empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

# ------------------------------------------------------------
# Binary training DTM
# ------------------------------------------------------------

binary_train <- dtm_train

binary_train$v[] <- 1

# ------------------------------------------------------------
# Term document frequencies
# ------------------------------------------------------------

term_document_frequency <- (
    slam::col_sums(
        binary_train
    )
)

# ------------------------------------------------------------
# Term co-occurrence matrix
#
# C[i,j] = number of training documents in which
# terms i and j occur together.
# ------------------------------------------------------------

term_cooccurrence <- (
    slam::crossprod_simple_triplet_matrix(
        binary_train
    )
)

# ------------------------------------------------------------
# Semantic coherence
#
# Mimno-style coherence:
#
# sum_{m=2}^{M}
# sum_{l=1}^{m-1}
# log(
#     (D(v_m, v_l) + 1)
#     / D(v_l)
# )
# ------------------------------------------------------------

calculate_coherence <- function(
    top_indices,
    cooccurrence,
    term_df
) {

    score <- 0

    if (length(top_indices) < 2) {
        return(score)
    }

    for (m in 2:length(top_indices)) {

        for (l in 1:(m - 1)) {

            word_m <- top_indices[m]
            word_l <- top_indices[l]

            cooccur <- (
                cooccurrence[
                    word_m,
                    word_l
                ]
            )

            denominator <- (
                term_df[
                    word_l
                ]
            )

            if (denominator > 0) {

                score <- score + log(
                    (
                        cooccur + 1
                    )
                    /
                    denominator
                )
            }
        }
    }

    return(score)
}

# ------------------------------------------------------------
# Cosine similarity
# ------------------------------------------------------------

cosine_similarity <- function(
    x,
    y
) {

    denominator <- (
        sqrt(
            sum(
                x^2
            )
        )
        *
        sqrt(
            sum(
                y^2
            )
        )
    )

    if (denominator == 0) {
        return(0)
    }

    return(
        sum(
            x * y
        )
        /
        denominator
    )
}

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Output containers
# ------------------------------------------------------------

summary_results <- data.frame()

topic_results <- data.frame()

top_term_results <- data.frame()

# ------------------------------------------------------------
# Candidate models
# ------------------------------------------------------------

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    # --------------------------------------------------------
    # Fit LDA
    # --------------------------------------------------------

    model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = RANDOM_SEED,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    # --------------------------------------------------------
    # Held-out perplexity
    # --------------------------------------------------------

    heldout_perplexity <- (
        topicmodels::perplexity(
            model,
            newdata = dtm_test
        )
    )

    # --------------------------------------------------------
    # Topic-term probabilities
    # --------------------------------------------------------

    beta <- posterior(
        model
    )$terms

    # --------------------------------------------------------
    # Topic coherence
    # --------------------------------------------------------

    coherence_values <- numeric(
        k
    )

    for (topic in seq_len(k)) {

        top_indices <- order(
            beta[
                topic,
            ],
            decreasing = TRUE
        )[
            seq_len(
                min(
                    TOP_N,
                    ncol(beta)
                )
            )
        ]

        coherence_values[
            topic
        ] <- calculate_coherence(
            top_indices,
            term_cooccurrence,
            term_document_frequency
        )

        # ----------------------------------------------------
        # Save top terms
        # ----------------------------------------------------

        top_words <- colnames(
            beta
        )[
            top_indices
        ]

        for (
            rank in seq_along(
                top_words
            )
        ) {

            top_term_results <- rbind(
                top_term_results,
                data.frame(
                    K = k,
                    Topic = topic,
                    Rank = rank,
                    Term = top_words[
                        rank
                    ],
                    Beta = beta[
                        topic,
                        top_indices[
                            rank
                        ]
                    ]
                )
            )
        }
    }

    # --------------------------------------------------------
    # Pairwise topic similarity
    # --------------------------------------------------------

    similarity_values <- c()

    if (k > 1) {

        for (
            i in 1:(k - 1)
        ) {

            for (
                j in (i + 1):k
            ) {

                similarity_values <- c(
                    similarity_values,
                    cosine_similarity(
                        beta[
                            i,
                        ],
                        beta[
                            j,
                        ]
                    )
                )
            }
        }
    }

    # --------------------------------------------------------
    # Topic-level results
    # --------------------------------------------------------

    topic_results <- rbind(
        topic_results,
        data.frame(
            K = k,
            Topic = seq_len(k),
            Coherence =
                coherence_values
        )
    )

    # --------------------------------------------------------
    # Elapsed time
    # --------------------------------------------------------

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    # --------------------------------------------------------
    # Model-level summary
    # --------------------------------------------------------

    summary_results <- rbind(
        summary_results,
        data.frame(
            K = k,

            Perplexity =
                heldout_perplexity,

            Mean_Coherence =
                mean(
                    coherence_values
                ),

            Median_Coherence =
                median(
                    coherence_values
                ),

            Min_Coherence =
                min(
                    coherence_values
                ),

            Mean_Similarity =
                mean(
                    similarity_values
                ),

            Median_Similarity =
                median(
                    similarity_values
                ),

            Max_Similarity =
                max(
                    similarity_values
                ),

            Elapsed_seconds =
                elapsed
        )
    )

    # --------------------------------------------------------
    # Save incrementally
    # --------------------------------------------------------

    write.csv(
        summary_results,
        summary_file,
        row.names = FALSE
    )

    write.csv(
        topic_results,
        topic_file,
        row.names = FALSE
    )

    write.csv(
        top_term_results,
        top_terms_file,
        row.names = FALSE
    )

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------

    cat(
        sprintf(
            paste0(
                "perplexity = %.4f",
                " | mean coherence = %.4f",
                " | mean similarity = %.4f",
                " | %.2f s\n"
            ),
            heldout_perplexity,
            mean(
                coherence_values
            ),
            mean(
                similarity_values
            ),
            elapsed
        )
    )

    flush.console()
}
'''

R_CANDIDATE_EVALUATION_SCRIPT.write_text(
    r_candidate_evaluation_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_CANDIDATE_EVALUATION_SCRIPT
)

Created: m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\lda\candidate_evaluation_patent_policy.R


#### 5.4.2 Run Candidate-Model Evaluation

The candidate topic resolutions are now estimated independently for the
patent-cited and policy-cited corpora using longer Gibbs-sampling chains.

For each candidate value of \(K\), the analysis records held-out perplexity,
semantic coherence, and inter-topic similarity. These diagnostics are used
to examine whether improvements in predictive fit are accompanied by
meaningful and sufficiently distinct latent topics.

The candidate models use the same fixed training and held-out document
splits established during the initial topic-number search.

In [87]:
# ---------------------------------------------------------
# Candidate-model evaluation configuration
# ---------------------------------------------------------

K_CANDIDATES = [
    20,
    30,
    40,
    50,
]

CANDIDATE_BURN_IN = 500
CANDIDATE_ITERATIONS = 2000
CANDIDATE_THIN = 50
TOP_TERMS_COHERENCE = 10


print(
    "Candidate K values :",
    K_CANDIDATES
)

print(
    "Burn-in            :",
    CANDIDATE_BURN_IN
)

print(
    "Iterations         :",
    CANDIDATE_ITERATIONS
)

print(
    "Thin               :",
    CANDIDATE_THIN
)

print(
    "Top terms          :",
    TOP_TERMS_COHERENCE
)

Candidate K values : [20, 30, 40, 50]
Burn-in            : 500
Iterations         : 2000
Thin               : 50
Top terms          : 10


In [88]:
candidate_results = {}
candidate_topic_diagnostics = {}
candidate_top_terms = {}


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Input files
    # -----------------------------------------------------

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    train_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_file = (
        LDA_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    # -----------------------------------------------------
    # Output files
    # -----------------------------------------------------

    summary_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_evaluation.csv"
    )

    topic_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_topic_diagnostics.csv"
    )

    top_terms_file = (
        LDA_DIR
        / f"{corpus_name}_candidate_top_terms.csv"
    )

    min_df = (
        final_min_doc_freq[
            corpus_name
        ]
    )

    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------

    print()
    print(
        "Running candidate evaluation..."
    )

    print(
        "Corpus     :",
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "K values   :",
        K_CANDIDATES
    )

    print(
        "Minimum DF :",
        min_df
    )

    # -----------------------------------------------------
    # Run R
    # -----------------------------------------------------

    candidate_run = subprocess.run(
        [
            str(RSCRIPT),
            str(
                R_CANDIDATE_EVALUATION_SCRIPT
            ),
            str(processed_file),
            str(train_file),
            str(test_file),
            str(summary_file),
            str(topic_file),
            str(top_terms_file),
            str(min_df),
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print(
        "Return code:",
        candidate_run.returncode
    )

    if candidate_run.stdout.strip():

        print()
        print("R output:")

        print(
            candidate_run.stdout
        )

    if candidate_run.stderr.strip():

        print()
        print("R messages:")

        print(
            candidate_run.stderr
        )

    if candidate_run.returncode != 0:

        raise RuntimeError(
            f"Candidate evaluation failed "
            f"for {corpus_name}."
        )

    # -----------------------------------------------------
    # Load results
    # -----------------------------------------------------

    candidate_results[
        corpus_name
    ] = pd.read_csv(
        summary_file
    )

    candidate_topic_diagnostics[
        corpus_name
    ] = pd.read_csv(
        topic_file
    )

    candidate_top_terms[
        corpus_name
    ] = pd.read_csv(
        top_terms_file
    )

    # -----------------------------------------------------
    # Display model-level results
    # -----------------------------------------------------

    print()

    display(
        candidate_results[
            corpus_name
        ]
    )


Running candidate evaluation...
Corpus     : PATENT CITED
K values   : [20, 30, 40, 50]
Minimum DF : 8
Return code: 0

R output:
Training documents: 622 
Held-out documents: 156 
Vocabulary: 1295 
Minimum DF: 8 
Empty training removed: 0 
Empty held-out removed: 0 

Fitting K = 20 ... perplexity = 498.7014 | mean coherence = -67.3228 | mean similarity = 0.0417 | 16.29 s
Fitting K = 30 ... perplexity = 482.4210 | mean coherence = -70.5756 | mean similarity = 0.0448 | 21.44 s
Fitting K = 40 ... perplexity = 466.1236 | mean coherence = -73.5502 | mean similarity = 0.0418 | 28.93 s
Fitting K = 50 ... perplexity = 456.8233 | mean coherence = -75.3984 | mean similarity = 0.0397 | 64.41 s


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 
4: package 'topicmodels' was built under R version 4.5.3 




,K,Perplexity,Mean_Coherence,Median_Coherence,Min_Coherence,Mean_Similarity,Median_Similarity,Max_Similarity,Elapsed_seconds
0,20,498.701427,-67.322782,-69.298065,-90.206575,0.041738,0.028200,0.367928,16.285177
1,30,482.421013,-70.575573,-70.732801,-90.153923,0.044840,0.025342,0.597921,21.436563
2,40,466.123642,-73.550158,-72.615672,-99.514910,0.041849,0.023937,0.676577,28.932323
3,50,456.823276,-75.398433,-75.827026,-103.063664,0.039749,0.020907,0.693252,64.407196



Running candidate evaluation...
Corpus     : POLICY CITED
K values   : [20, 30, 40, 50]
Minimum DF : 4
Return code: 0

R output:
Training documents: 313 
Held-out documents: 79 
Vocabulary: 1371 
Minimum DF: 4 
Empty training removed: 0 
Empty held-out removed: 0 

Fitting K = 20 ... perplexity = 556.6528 | mean coherence = -63.5914 | mean similarity = 0.0443 | 9.17 s
Fitting K = 30 ... perplexity = 542.9512 | mean coherence = -63.9442 | mean similarity = 0.0419 | 21.89 s
Fitting K = 40 ... perplexity = 525.5769 | mean coherence = -68.5028 | mean similarity = 0.0398 | 36.91 s
Fitting K = 50 ... perplexity = 520.5469 | mean coherence = -70.4710 | mean similarity = 0.0393 | 21.87 s


R messages:
Warning messages:
1: package 'tm' was built under R version 4.5.3 
2: package 'NLP' was built under R version 4.5.3 
3: package 'slam' was built under R version 4.5.3 
4: package 'topicmodels' was built under R version 4.5.3 




,K,Perplexity,Mean_Coherence,Median_Coherence,Min_Coherence,Mean_Similarity,Median_Similarity,Max_Similarity,Elapsed_seconds
0,20,556.652838,-63.591437,-64.474449,-79.015144,0.044296,0.030319,0.269884,9.171121
1,30,542.951173,-63.944172,-64.882839,-94.018184,0.041856,0.029337,0.331223,21.887099
2,40,525.576888,-68.502849,-66.651181,-91.478295,0.039787,0.026580,0.745435,36.913572
3,50,520.546930,-70.470997,-70.799635,-92.151446,0.039326,0.025708,0.628700,21.872864


## 5.5 Save Model-Selection Outputs

All model-selection outputs are saved to disk before proceeding to topic
interpretation.

This separates the computationally intensive LDA estimation stage from the
subsequent interpretive analysis. The saved outputs allow topic-number
selection, diagnostic visualization, and manual topic inspection to be
performed in a separate notebook without rerunning preprocessing or candidate
LDA estimation.

For each corpus, the saved outputs include:

- coarse topic-number search results;
- extended topic-number search results;
- candidate-model diagnostics;
- topic-level coherence diagnostics;
- top terms and topic-word probabilities for each candidate model; and
- fixed training and held-out document indices.

These files constitute the model-selection outputs used in the subsequent
topic-interpretation workflow.

In [89]:
# ---------------------------------------------------------
# Save / verify all model-selection outputs
# ---------------------------------------------------------

MODEL_SELECTION_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "model_selection"
)

MODEL_SELECTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


for corpus_name in [
    "patent_cited",
    "policy_cited",
]:

    # -----------------------------------------------------
    # Candidate evaluation summary
    # -----------------------------------------------------

    candidate_summary_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_candidate_evaluation.csv"
    )

    candidate_results[
        corpus_name
    ].to_csv(
        candidate_summary_file,
        index=False,
    )

    # -----------------------------------------------------
    # Topic-level coherence
    # -----------------------------------------------------

    topic_diagnostics_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_candidate_topic_diagnostics.csv"
    )

    candidate_topic_diagnostics[
        corpus_name
    ].to_csv(
        topic_diagnostics_file,
        index=False,
    )

    # -----------------------------------------------------
    # Candidate top terms
    # -----------------------------------------------------

    top_terms_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_candidate_top_terms.csv"
    )

    candidate_top_terms[
        corpus_name
    ].to_csv(
        top_terms_file,
        index=False,
    )

    # -----------------------------------------------------
    # Coarse K search
    # -----------------------------------------------------

    coarse_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_coarse_k_search.csv"
    )

    coarse_results[
        corpus_name
    ].to_csv(
        coarse_file,
        index=False,
    )

    # -----------------------------------------------------
    # Extended K search
    # -----------------------------------------------------

    extended_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_extended_k_search.csv"
    )

    extended_results[
        corpus_name
    ].to_csv(
        extended_file,
        index=False,
    )

    # -----------------------------------------------------
    # Combined K search
    # -----------------------------------------------------

    combined = (
        pd.concat(
            [
                coarse_results[
                    corpus_name
                ],
                extended_results[
                    corpus_name
                ],
            ],
            ignore_index=True,
        )
        .drop_duplicates(
            subset="K",
            keep="last",
        )
        .sort_values("K")
        .reset_index(drop=True)
    )

    combined_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_combined_k_search.csv"
    )

    combined.to_csv(
        combined_file,
        index=False,
    )

    # -----------------------------------------------------
    # Train / test indices
    # -----------------------------------------------------

    train_indices_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    test_indices_file = (
        MODEL_SELECTION_DIR
        / f"{corpus_name}_test_indices.csv"
    )

    pd.DataFrame({
        "train_id":
            train_test_indices[
                corpus_name
            ]["train"]
    }).to_csv(
        train_indices_file,
        index=False,
    )

    pd.DataFrame({
        "test_id":
            train_test_indices[
                corpus_name
            ]["test"]
    }).to_csv(
        test_indices_file,
        index=False,
    )

    # -----------------------------------------------------
    # Diagnostics
    # -----------------------------------------------------

    print()
    print(
        corpus_name
        .replace("_", " ")
        .upper()
    )

    print(
        "-" * 50
    )

    print(
        "Candidate models :",
        len(
            candidate_results[
                corpus_name
            ]
        )
    )

    print(
        "Topic diagnostics:",
        len(
            candidate_topic_diagnostics[
                corpus_name
            ]
        )
    )

    print(
        "Top-term rows    :",
        len(
            candidate_top_terms[
                corpus_name
            ]
        )
    )

    print(
        "Saved to         :",
        MODEL_SELECTION_DIR
    )


PATENT CITED
--------------------------------------------------
Candidate models : 4
Topic diagnostics: 140
Top-term rows    : 1400
Saved to         : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\model_selection

POLICY CITED
--------------------------------------------------
Candidate models : 4
Topic diagnostics: 140
Top-term rows    : 1400
Saved to         : m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\model_selection


In [90]:
MODEL_SELECTION_EXCEL = (
    MODEL_SELECTION_DIR
    / "patent_policy_model_selection.xlsx"
)


with pd.ExcelWriter(
    MODEL_SELECTION_EXCEL,
    engine="openpyxl",
) as writer:

    for corpus_name in [
        "patent_cited",
        "policy_cited",
    ]:

        candidate_results[
            corpus_name
        ].to_excel(
            writer,
            sheet_name=(
                f"{corpus_name}_models"
            )[:31],
            index=False,
        )

        candidate_topic_diagnostics[
            corpus_name
        ].to_excel(
            writer,
            sheet_name=(
                f"{corpus_name}_topics"
            )[:31],
            index=False,
        )

        candidate_top_terms[
            corpus_name
        ].to_excel(
            writer,
            sheet_name=(
                f"{corpus_name}_terms"
            )[:31],
            index=False,
        )


print(
    "Saved:",
    MODEL_SELECTION_EXCEL
)

Saved: m:\projects_latex\review_paper_2\output\patent_policy_topic_modeling\model_selection\patent_policy_model_selection.xlsx
